## RAG 예제: ollama + gemma2 + Chroma를 활용

In [ ]:
# 사전설치 : pip install langchain langchain-community chromadb sentence-transformers langchain-text-splitters
import os
import shutil  # 오래된 벡터 저장소를 삭제하기 위해 임포트
# RAG 체인 구성에 필요한 핵심 모듈 임포트
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# 나머지 모듈 임포트
from langchain_community.document_loaders import TextLoader
from langchain_community.vectorstores import Chroma
# from langchain_community.embeddings import SentenceTransformerEmbeddings
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.llms import Ollama
from langchain_text_splitters import RecursiveCharacterTextSplitter  # 문맥이 가능한 한 유지되도록 청크 분할(문단, 줄바꿈, 공백 등)

# 1. 설정
# ----------------------------------------------------------------------
# 로드할 .txt 파일 경로 (요청 경로 반영)
TEXT_FILE_PATH = "./dataset/history.txt"
# ChromaDB를 저장할 로컬 디렉토리 경로 (Windows 로컬 PC에 생성됨)
PERSIST_DIRECTORY = "./chroma_store"
OLLAMA_BASE_URL = "http://localhost:11434"
OLLAMA_MODEL_NAME = "gemma2"

In [ ]:
# 2. 문서 로드 및 청크 분할
print(f"1. '{TEXT_FILE_PATH}' 한글 파일 로드 중...")
try:
    loader = TextLoader(TEXT_FILE_PATH, encoding='UTF8')
    documents = loader.load()
except FileNotFoundError:
    print(f"오류: 파일이 존재하지 않습니다. 경로를 확인하세요: {TEXT_FILE_PATH}")
    exit()
except Exception as e:
    print(f"파일 로드 중 오류 발생: {e}")
    exit()

text_splitter = RecursiveCharacterTextSplitter(
    # 청크 크기를 늘려 더 많은 컨텍스트가 포함되도록 함
    chunk_size=500,
    chunk_overlap=50,
    length_function=len
)
texts = text_splitter.split_documents(documents)
print(f"-> 총 {len(texts)}개의 청크로 분할되었습니다.")

In [ ]:
# 3. 임베딩 모델 정의
print(f"\n2. 임베딩 모델 정의")
# 한글에 특화된 HuggingFace 임베딩 모델을 로드합니다.
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

In [ ]:
# 4. ChromaDB 벡터 저장소 생성
# ----------------------------------------------------------------------
if os.path.exists(PERSIST_DIRECTORY):
    print(f"'{PERSIST_DIRECTORY}' (기존 벡터 저장소) 삭제 중...")
    shutil.rmtree(PERSIST_DIRECTORY)

print(f"\n3. ChromaDB 벡터 저장소 생성 및 '{PERSIST_DIRECTORY}'에 저장 중...")
vectordb = Chroma.from_documents(
    documents=texts,
    embedding=embeddings,
    persist_directory=PERSIST_DIRECTORY
)
vectordb.persist()
print(f"-> 벡터 저장소가 성공적으로 저장되었습니다.")

In [ ]:
# 5. Ollama LLM 및 검색기(Retriever) 정의
print(f"\n4. Ollama LLM ({OLLAMA_MODEL_NAME}) 및 검색기 초기화 중...")

try:
    llm = Ollama(model=OLLAMA_MODEL_NAME, base_url=OLLAMA_BASE_URL)
    llm.invoke("Hello") # 연결 테스트
    print("-> Ollama LLM 연결 성공.")
except Exception as e:
    print(f"오류: Ollama LLM에 연결할 수 없습니다. (URL: {OLLAMA_BASE_URL})")
    print(f"Ollama가 실행 중인지, '{OLLAMA_MODEL_NAME}' 모델이 설치되었는지 확인하세요.")
    exit()

# 벡터 저장소를 검색기(Retriever)로 변환
retriever = vectordb.as_retriever()

In [ ]:
# 6. RAG 체인 구성 (LCEL 방식)
print("\n5. RAG 체인 구성 (LCEL 방식)...")

template = """
당신은 질문에 답변하는 AI 어시스턴트입니다.
제공된 context만을 바탕으로 질문에 답변하세요. 모르면 모른다고 답하세요.

[Context]
{context}

[Question]
{question}
"""
prompt = ChatPromptTemplate.from_template(template)

rag_chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

In [ ]:
# 7. RAG 체인 실행 (질의)
print("\n6. RAG 질의 실행...")
query = "고조선은 언제 설립되었는지 알려줘."

try:
    response = rag_chain.invoke(query)

    print("\n[질문]:", query)
    print("[답변]:", response)

except Exception as e:
    print(f"RAG 체인 실행 중 오류가 발생했습니다: {e}")